# Dự án Nhận diện Thủ ngữ (Sign Language Recognition) bằng YOLO26

## 1. Giới thiệu dự án
Dự án này tập trung vào việc xây dựng và huấn luyện một mô hình học sâu để nhận diện các ký hiệu thủ ngữ (Sign Language). Mô hình được sử dụng trong notebook này là **YOLO26**, nhằm mục đích phát hiện và phân loại các cử chỉ tay theo thời gian thực với tốc độ và độ chính xác cao.

## 2. Mục tiêu
- Tiền xử lý và chuẩn bị tập dữ liệu (dataset) thủ ngữ để đưa vào huấn luyện.
- Khởi tạo và cấu hình mô hình nhận diện YOLO26.
- Huấn luyện (Train) mô hình để nhận diện chính xác các từ vựng hoặc chữ cái thủ ngữ.
- Đánh giá (Evaluate) hiệu suất của mô hình thông qua các độ đo cơ bản như mAP, Precision, Recall.
- Thử nghiệm (Inference) mô hình trên hình ảnh hoặc video thực tế.

## 3. Cấu trúc Notebook
1. **Cài đặt và Import thư viện:** Thiết lập môi trường và tải các thư viện cần thiết (ví dụ: opencv, pytorch, matplotlib, các thư viện hỗ trợ YOLO26...).
2. **Khám phá và Tiền xử lý dữ liệu (EDA):** Tải bộ dữ liệu, kiểm tra số lượng ảnh, label format, và phân chia train/val/test.
3. **Cấu hình YOLO26:** Tạo file `data.yaml` cấu hình đường dẫn và thông tin số lượng class (nhãn dán) cho thủ ngữ.
4. **Huấn luyện mô hình (Training):** Thực thi tiến trình train YOLO26 trên dữ liệu đã chuẩn bị.
5. **Kiểm tra và Trực quan hóa (Validation & Visualization):** Xem các biểu đồ loss, metrics và so sánh kết quả ground-truth với dự đoán (predictions).

## 4. Cài đặt môi trường (Colab + GPU)
Dataset là **Sign Language MNIST** (ảnh xám 28×28 dạng CSV) → bài toán **phân loại (classification)**,
dùng `yolo26n-cls.pt`. Cài thư viện và kiểm tra GPU.

In [ ]:
!pip install -q ultralytics kagglehub

import ultralytics, torch
ultralytics.checks()
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 5. Tải dataset bằng kagglehub
`kagglehub` tự tải & cache, không cần `kaggle.json`.

In [ ]:
import kagglehub

DATA_PATH = kagglehub.dataset_download("datamunge/sign-language-mnist")
print("Path to dataset files:", DATA_PATH)

from pathlib import Path
for p in sorted(Path(DATA_PATH).rglob("*.csv")):
    print(" -", p.name)

## 6. Khám phá dữ liệu (EDA)
Mỗi dòng CSV = 1 nhãn (`label`) + 784 pixel (28×28). Nhãn 0–25, **bỏ 9 (J) và 25 (Z)** ⇒ 24 lớp.

In [ ]:
import pandas as pd
from pathlib import Path

def find_csv(keyword):
    hits = [p for p in Path(DATA_PATH).rglob("*.csv") if keyword in p.name.lower()]
    if not hits:
        raise FileNotFoundError(f"Không tìm thấy CSV chứa '{keyword}' trong {DATA_PATH}")
    return str(hits[0])

TRAIN_CSV = find_csv("train")
TEST_CSV = find_csv("test")

df = pd.read_csv(TRAIN_CSV)
print("Train shape:", df.shape, "| Test shape:", pd.read_csv(TEST_CSV).shape)
print("Số lớp (train):", df["label"].nunique())
print("Phân bố nhãn:")
print(df["label"].value_counts().sort_index())

## 7. Tiền xử lý: CSV → ImageFolder
YOLO classification cần cấu trúc thư mục `train/<Letter>/*.png` và `val/<Letter>/*.png`.
Chuyển mỗi dòng CSV thành ảnh PNG 28×28, đặt vào thư mục theo chữ cái (0→A, 1→B, …).

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from pathlib import Path

DATASET_DIR = Path("/content/datasets/sign_language_mnist")

def label_to_letter(lab: int) -> str:
    return chr(ord("A") + int(lab))   # 0->A, 1->B, ... (J/Z không xuất hiện trong data)

def csv_to_imagefolder(csv_path: str, out_dir: Path) -> int:
    df = pd.read_csv(csv_path)
    labels = df["label"].to_numpy()
    imgs = df.drop(columns=["label"]).to_numpy(dtype="uint8").reshape(-1, 28, 28)
    for i, (lab, img) in enumerate(zip(labels, imgs)):
        d = out_dir / label_to_letter(lab)
        d.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img, mode="L").save(d / f"{i}.png")
    return len(df)

# Train từ train.csv, val từ test.csv (split sẵn của dataset)
if not (DATASET_DIR / "train").exists():
    n_tr = csv_to_imagefolder(TRAIN_CSV, DATASET_DIR / "train")
    n_va = csv_to_imagefolder(TEST_CSV, DATASET_DIR / "val")
    print(f"Đã tạo {n_tr} ảnh train, {n_va} ảnh val tại {DATASET_DIR}")
else:
    print("Đã tồn tại, bỏ qua chuyển đổi:", DATASET_DIR)

classes = sorted(p.name for p in (DATASET_DIR / "train").iterdir() if p.is_dir())
print("Các lớp:", classes, "| Tổng:", len(classes))

## 8. Khởi tạo mô hình YOLO26n-cls
Tải pretrained classification weights (ImageNet) làm điểm khởi đầu.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-cls.pt")   # nano, classification
model.info()

## 9. Huấn luyện (Training)
Ảnh gốc 28×28; đặt `imgsz=64` (upscale nhẹ) cho ổn định. Tự chọn GPU nếu có.

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

results = model.train(
    data=str(DATASET_DIR),       # classification: trỏ vào thư mục chứa train/ val/
    epochs=30,
    imgsz=64,
    batch=64,
    device=DEVICE,
    patience=10,                 # early stopping
    project="runs/sign_language",
    name="yolo26n_cls",
)
print("Kết quả lưu tại:", results.save_dir)

## 10. Đánh giá & Trực quan hóa

In [ ]:
metrics = model.val()
print("Top-1 accuracy:", round(metrics.top1, 4))
print("Top-5 accuracy:", round(metrics.top5, 4))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

save_dir = Path(results.save_dir)
for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = save_dir / name
    if p.exists():
        print(p)
        display(Image(str(p)))

In [ ]:
# Inference thử trên 1 ảnh trong tập val
from pathlib import Path

sample = next((DATASET_DIR / "val").rglob("*.png"))
pred = model.predict(sample, imgsz=64, device=DEVICE)[0]
print("Ảnh:", sample)
print("Dự đoán:", pred.names[pred.probs.top1], "| confidence:", round(float(pred.probs.top1conf), 4))